## Load

In [7]:
# output directory of main.py
OUTDIR = "/projects/wangc/m344313/OVTMA_project/output/bms1_interface_radius_25_power"  
GRAPH_TYPE = "radius"   

roi_col       = "roi_id"        
subject_col   = "patient_id"    # patient/subject identifier (if absent, set to None)
label_col     = "roi_label"     # ROI-level responder label (0/1). If only patient labels exist, map them onto ROIs.
type_col      = "phenotype"     

# If you have only patient-level labels, set label_col to that name and we’ll broadcast later.

# === Graph load paths ===
import os, pickle, pandas as pd, networkx as nx, numpy as np, warnings, math
from pathlib import Path

graphs_dir = Path(OUTDIR) / "graphs"
df_path    = Path(OUTDIR) / "dataframes" / "df.csv"
G_all_pkl  = graphs_dir / f"G_all_{GRAPH_TYPE}.pkl"
graph_dict_pkl = graphs_dir / f"graph_dict_{GRAPH_TYPE}.pkl"

assert df_path.exists(), f"Missing df at {df_path}"
assert G_all_pkl.exists() or graph_dict_pkl.exists(), "Missing saved graphs. Re-run main.py to generate."

df = pd.read_csv(df_path)
# If subject_col missing, create a dummy subject group per ROI (so GroupKFold still works)
if subject_col not in df.columns:
    subject_col = "subject_id"
    df[subject_col] = df[roi_col].astype(str).map(lambda r: r.split("_")[0])

# Load graphs
if Path(graph_dict_pkl).exists():
    with open(graph_dict_pkl, "rb") as f:
        graph_dict = pickle.load(f)
else:
    with open(G_all_pkl, "rb") as f:
        G_all = pickle.load(f)
    # Split by connected components (each is an ROI island with node names (roi, node_id))
    graph_dict = {}
    for C in nx.connected_components(G_all):
        sub = G_all.subgraph(C).copy()
        # Name it by ROI taken from tuple node (roi, node_id)
        any_node = next(iter(sub.nodes))
        roi = any_node[0] if isinstance(any_node, tuple) else "unknown_roi"
        # Relabel nodes back to original node_id if needed
        if isinstance(any_node, tuple):
            sub = nx.relabel_nodes(sub, lambda t: t[1])
        graph_dict[roi] = sub

len(graph_dict), list(graph_dict)[:3]


(219, ['Mel30_001', 'Mel30_002', 'Mel30_003'])

In [8]:
# Ensure node type attribute "label" is set from df[type_col]
# df must have columns: roi_id, cell_id, type_col, label_col, subject_col
need_cols = {roi_col, "cell_id", type_col}
missing = need_cols - set(df.columns)
if missing:
    raise ValueError(f"df.csv missing columns: {missing}")

# Build a lookup for each ROI: cell_id -> categorical type
cat_map = {}
for roi, sub in df[[roi_col, "cell_id", type_col]].dropna().astype({roi_col:str}).groupby(roi_col):
    cat_map[str(roi)] = dict(zip(sub["cell_id"].astype(int).values, sub[type_col].astype(str).values))

# Attach node label attribute
for roi, G in graph_dict.items():
    cmap = cat_map.get(str(roi), {})
    nx.set_node_attributes(G, {n: cmap.get(int(n), "UNK") for n in G.nodes()}, name="label")

# ROI label y and patient group
# If ROI labels are missing but subject-level labels exist, broadcast subject->ROI
if label_col not in df.columns and "subject_label" in df.columns:
    print("Broadcasting subject_label to ROI label...")
    label_col = "subject_label"

roi_to_label = (
    df[[roi_col, label_col]]
    .dropna()
    .drop_duplicates(subset=[roi_col])
    .set_index(roi_col)[label_col]
    .astype(int)
    .to_dict()
)

roi_to_subject = (
    df[[roi_col, subject_col]]
    .dropna()
    .drop_duplicates(subset=[roi_col])
    .set_index(roi_col)[subject_col]
    .astype(str)
    .to_dict()
)

# Filter to ROIs with labels
roi_list = [r for r in graph_dict if r in roi_to_label]
X_graphs = [graph_dict[r] for r in roi_list]
y = np.array([roi_to_label[r] for r in roi_list], dtype=int)
groups = np.array([roi_to_subject[r] for r in roi_list])

print(f"ROIs usable: {len(roi_list)} / {len(graph_dict)}")
print("Class balance:", {int(k): int((y==k).sum()) for k in np.unique(y)})
print("Patients:", len(np.unique(groups)))


ROIs usable: 219 / 219
Class balance: {0: 139, 1: 80}
Patients: 21


## Graph kernel path: WL subtree kernel + SVM

In [9]:
from grakel import graph_from_networkx
from grakel.kernels import WeisfeilerLehman, VertexHistogram
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
from joblib import Parallel, delayed

# Convert to GraKeL format: expects node attribute 'label'
gk_graphs = list(graph_from_networkx(
    X_graphs,
    node_labels_tag="label",
    # We pass no edge labels; kernel will use unlabeled edges + labeled vertices
))

# Build WL kernel: WL (h iterations) with base kernel = VertexHistogram
h = 4  # try 1–4
wl = WeisfeilerLehman(n_iter=h, base_graph_kernel=VertexHistogram, normalize=True)
K = wl.fit_transform(gk_graphs)   # Gram matrix (N x N)
print("Kernel matrix shape:", K.shape)

# predict with Logistic Regression
cv = StratifiedGroupKFold(n_splits=5)
y_scores, y_true = [], []

for tr, te in cv.split(np.arange(len(y)), y, groups):
    K_tr = K[np.ix_(tr, tr)]
    K_te = K[np.ix_(te, tr)]
    clf = LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs", max_iter=2000, class_weight="balanced"
    )
    clf.fit(K_tr, y[tr])
    proba = clf.predict_proba(K_te)[:, 1]
    y_scores.extend(proba)
    y_true.extend(y[te])

auc = roc_auc_score(y_true, y_scores)
acc = accuracy_score(y_true, (np.array(y_scores) >= 0.5).astype(int))
print(f"[WL kernel + Logistic] AUC={auc:.3f}  ACC={acc:.3f}")


Kernel matrix shape: (219, 219)
[WL kernel + Logistic] AUC=0.677  ACC=0.612


In [10]:
from sklearn.linear_model import LogisticRegression
from pathlib import Path
import numpy as np, pandas as pd, joblib

# === Fit logistic regression on full WL kernel ===
clf_full = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=4000,
    class_weight="balanced"
)
clf_full.fit(K, y)

# === Coefficients in kernel (dual) space ===
# For kernel logistic regression, coefficients act on support of all training samples
dual = clf_full.coef_.ravel()  # one weight per training sample (same dim as y)

# Rank ROIs by coefficient magnitude and sign
pos_mask = (y == 1)
neg_mask = (y == 0)
top_pos = np.argsort(-dual * pos_mask)[:10]
top_neg = np.argsort(dual * neg_mask)[:10]

print("Top + prototypes (responders):", [roi_list[i] for i in top_pos])
print("Top − prototypes (non-responders):", [roi_list[i] for i in top_neg])

# === Save artifacts ===
outK = Path(OUTDIR) / "subgraph" / "wl_kernel_logreg"
outK.mkdir(parents=True, exist_ok=True)

np.save(outK / "K_wl.npy", K)
pd.DataFrame({"roi": roi_list, "y": y, "group": groups}).to_csv(outK / "roi_meta.csv", index=False)

joblib.dump({"wl": wl, "clf": clf_full, "roi_list": roi_list}, outK / "model.pkl")

print("Saved kernel, ROI meta, and model to:", outK)


Top + prototypes (responders): ['Mel52_002', 'Mel52_025', 'Mel40_007', 'Mel52_026', 'Mel40_006', 'Mel52_028', 'Mel52_001', 'Mel50_010', 'Mel40_005', 'Mel50_016']
Top − prototypes (non-responders): ['Mel35_002', 'Mel43_007', 'Mel35_005', 'Mel35_010', 'Mel35_006', 'Mel35_007', 'Mel35_001', 'Mel30_004', 'Mel30_008', 'Mel35_009']
Saved kernel, ROI meta, and model to: /projects/wangc/m344313/OVTMA_project/output/bms1_interface_radius_25_power/subgraph/wl_kernel_logreg


## Small colored-graphlet mining (3–4 nodes) + elastic-net

In [ ]:
'''
python graphlet_miner.py \
--graphs-pkl /projects/wangc/m344313/OVTMA_project/output/bms1_interface_radius_25_power/graphs/graph_dict_radius.pkl \
--df /projects/wangc/m344313/OVTMA_project/output/bms1_interface_radius_25_power/dataframes/df.csv \
--roi-col roi_id --subject-col patient_id --label-col roi_label \
--type-col phenotype \
--k 3 4 \
--max-samples-per-k 6000 \
--mode combos \
--normalize by_total \
--n-jobs 1 \
--outdir /projects/wangc/m344313/OVTMA_project/output/bms1_interface_radius_25_power/subgraph/graphlet_mining
'''

In [ ]:
# === Load graphlet-miner artifacts so the eval cells can run ===
from pathlib import Path
import json, numpy as np, pandas as pd
from scipy.sparse import load_npz

# ---- Configure where your miner wrote files ----
# If you followed my earlier paths, try either ".../subgraph/graphlet_mining" or ".../evaluate/graphlet_mining".
OUTDIR = Path(OUTDIR)  # reuse if already set above; else set a string path here
CANDIDATES = [
    OUTDIR / "subgraph" / "graphlet_mining",
    OUTDIR / "evaluate" / "graphlet_mining",
]
MINER_OUTDIR = next((p for p in CANDIDATES if p.exists()), None)
assert MINER_OUTDIR is not None, f"Could not find miner outputs under: {CANDIDATES}"

print(f"[load] Using miner outputs at: {MINER_OUTDIR}")

# ---- Required files produced by graphlet_miner.py ----
X_path         = MINER_OUTDIR / "X_graphlets.npz"
roi_list_path  = MINER_OUTDIR / "roi_list.json"
vocab_path     = MINER_OUTDIR / "vocab.json"
roi_meta_path  = MINER_OUTDIR / "roi_meta.csv"
token_meta_path= MINER_OUTDIR / "token_meta.json"

for p in [X_path, roi_list_path, vocab_path, roi_meta_path]:
    assert p.exists(), f"Missing required file: {p}"

# ---- Load artifacts ----
X = load_npz(X_path)                           # (n_rois × n_motifs) CSR
roi_list = json.loads(roi_list_path.read_text())
vocab    = json.loads(vocab_path.read_text())
roi_meta = pd.read_csv(roi_meta_path)

# Optional (nice to have for interpretability & k_list inference)
token_meta = {}
if token_meta_path.exists():
    token_meta = json.loads(token_meta_path.read_text())

# ---- Build y and groups in the same order as roi_list ----
# Expect columns: roi, y, group
need_cols = {"roi", "y", "group"}
missing = need_cols - set(roi_meta.columns)
assert not missing, f"roi_meta.csv missing columns: {missing}"

# Align to roi_list order
roi_to_y     = dict(zip(roi_meta["roi"].astype(str),   roi_meta["y"].astype(int)))
roi_to_group = dict(zip(roi_meta["roi"].astype(str),   roi_meta["group"].astype(str)))

y = np.array([roi_to_y[str(r)] for r in roi_list], dtype=int)
groups = np.array([roi_to_group[str(r)] for r in roi_list], dtype=str)

# ---- Token index map ----
tok2idx = {t:i for i, t in enumerate(vocab)}

# ---- Infer k_list for the pretty print (from token_meta if present) ----
if token_meta:
    k_list = sorted({ int(m.get("k", -1)) for m in token_meta.values() if "k" in m })
else:
    k_list = ["3/4"]  # fallback label if token_meta wasn't saved

# You likely sampled during mining; if you want a label for the printout:
max_samples_per_k = None  # unknown from artifacts; set for display only

# ---- Sanity prints ----
print(f"[load] X shape: {X.shape}, nnz={X.nnz}")
print(f"[load] #ROIs: {len(roi_list)}, #motifs: {len(vocab)}")
print(f"[load] Class balance: {{0: {(y==0).sum()}, 1: {(y==1).sum()}}}")
print(f"[load] Patients (groups): {len(np.unique(groups))}")
print(f"[load] k_list (inferred): {k_list}")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score 


cv = StratifiedGroupKFold(n_splits=min(5, len(np.unique(groups))))
auc_list, acc_list = [], []

for tr, te in cv.split(X, y, groups):
    clf = make_pipeline(
        StandardScaler(with_mean=False),  # for sparse
        LogisticRegression(
            penalty="elasticnet", solver="saga", l1_ratio=0.5,
            class_weight="balanced", max_iter=5000, C=1.0, n_jobs=-1
        )
    )
    clf.fit(X[tr], y[tr])
    proba = clf.predict_proba(X[te])[:,1]
    auc_list.append(roc_auc_score(y[te], proba))
    acc_list.append(accuracy_score(y[te], (proba>=0.5).astype(int)))

print(f"[Graphlet mining] Grouped CV AUC={np.mean(auc_list):.3f}±{np.std(auc_list):.3f}  "
      f"ACC={np.mean(acc_list):.3f}±{np.std(acc_list):.3f}  "
      f"(k={k_list}, samples/k={max_samples_per_k})")


In [ ]:
from sklearn.pipeline import Pipeline

final_clf: Pipeline = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5,
        class_weight="balanced", max_iter=8000, C=1.0, n_jobs=-1
    )
)
final_clf.fit(X, y)

# Get non-zero coefficients
LR = final_clf.named_steps["logisticregression"]
coef = LR.coef_.ravel()
nz_idx = np.where(coef != 0)[0]
order = nz_idx[np.argsort(-np.abs(coef[nz_idx]))]

def pretty_meta(tok):
    # unpack meta tuple back from the hash if you cached it; here we only have hash string
    # quick sketch: we can re-materialize one instance from roi_to_counts keys if needed,
    # but for now just report hash. If you want human-readable, store 'meta' alongside hash in Cell 8.
    return tok

topK = 30
rows = []
for i in order[:topK]:
    rows.append({"motif_hash": vocab[i], "coef": float(coef[i])})

top_df = pd.DataFrame(rows)
display(top_df)

# Save artifacts
outM = Path(OUTDIR) / "subgraph" / "graphlet_mining"
outM.mkdir(parents=True, exist_ok=True)
import joblib, json
joblib.dump({"tok2idx": tok2idx, "vocab": vocab, "model": final_clf, "roi_list": roi_list}, outM / "model.pkl")
pd.DataFrame({"roi": roi_list, "y": y, "group": groups}).to_csv(outM / "roi_meta.csv", index=False)
print("Saved model and ROI meta to:", outM)


In [ ]:
# For each top motif, find ROIs where it is most abundant
def top_rois_for_token(tok_hash, k=5):
    j = tok2idx[tok_hash]
    scores = X[:, j].toarray().ravel()
    idx = np.argsort(-scores)[:k]
    return [(roi_list[i], scores[i]) for i in idx if scores[i] > 0]

for _, r in top_df.head(10).iterrows():
    print("\nMotif:", r["motif_hash"], "coef:", f"{r['coef']:.3f}")
    print("  top ROIs:", top_rois_for_token(r["motif_hash"]))
